# WellCo Churn Prediction — Phase 2: Feature Engineering

Builds a single member-level feature frame for train and test from all four event tables.

**Output:** `train_features.parquet`, `test_features.parquet`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

DATA   = '../assignment_instructions/'
CUTOFF = pd.Timestamp('2025-07-15')   # outreach date; tenure anchor
OBS_START = pd.Timestamp('2025-07-01')
OBS_END_DAY = 14                       # Jul 14 = last event day

# --- Load train ---
labels    = pd.read_csv(DATA + 'train/churn_labels.csv',  parse_dates=['signup_date'])
app_tr    = pd.read_csv(DATA + 'train/app_usage.csv',     parse_dates=['timestamp'])
web_tr    = pd.read_csv(DATA + 'train/web_visits.csv',    parse_dates=['timestamp'])
claims_tr = pd.read_csv(DATA + 'train/claims.csv',        parse_dates=['diagnosis_date'])

# --- Load test ---
test      = pd.read_csv(DATA + 'test/test_members.csv',       parse_dates=['signup_date'])
app_te    = pd.read_csv(DATA + 'test/test_app_usage.csv',     parse_dates=['timestamp'])
web_te    = pd.read_csv(DATA + 'test/test_web_visits.csv',    parse_dates=['timestamp'])
claims_te = pd.read_csv(DATA + 'test/test_claims.csv',        parse_dates=['diagnosis_date'])

print('Train members:', len(labels), '| Test members:', len(test))

## 1. Tenure feature

In [ ]:
def build_tenure(df):
    return pd.DataFrame({
        'member_id':   df['member_id'],
        'tenure_days': (CUTOFF - df['signup_date']).dt.days
    })

tenure_tr = build_tenure(labels)
tenure_te = build_tenure(test)
print('tenure_days range (train):', tenure_tr['tenure_days'].min(), '->', tenure_tr['tenure_days'].max())

## 2. App usage features

Includes: session counts, active days, recency, 7-day and 3-day trend windows, and variance features (std, CV, max daily, weekend ratio).

In [ ]:
def build_app_features(app, spine):
    app = app.copy()
    app['day'] = (app['timestamp'].dt.normalize() - OBS_START).dt.days + 1
    app['is_weekend'] = app['timestamp'].dt.dayofweek >= 5

    # Daily session counts per member (days 1-14)
    daily = app.groupby(['member_id', 'day']).size().reset_index(name='cnt')

    # Window aggregates for trend features
    def window_sum(d_from, d_to, name):
        return (daily[daily['day'].between(d_from, d_to)]
                .groupby('member_id')['cnt'].sum().rename(name))

    sess_d1_3  = window_sum(1,  3,  'sess_d1_3')
    sess_d4_6  = window_sum(4,  6,  'sess_d4_6')
    sess_d1_7  = window_sum(1,  7,  'sess_d1_7')
    sess_d8_14 = window_sum(8,  14, 'sess_d8_14')

    # Variance stats across all 14 days
    # Expand to full 14-day grid so absent days count as 0
    members = spine['member_id'].unique()
    full_grid = pd.MultiIndex.from_product([members, range(1, 15)],
                                           names=['member_id', 'day'])
    daily_full = (daily.set_index(['member_id', 'day'])
                       .reindex(full_grid, fill_value=0)
                       .reset_index())
    var_stats = daily_full.groupby('member_id')['cnt'].agg(
        session_std_daily='std',
        max_sessions_day='max'
    ).reset_index()

    # Summary totals
    total = app.groupby('member_id').agg(
        session_count      = ('day', 'count'),
        active_days_app    = ('day', 'nunique'),
        last_day           = ('day', 'max'),
        weekend_sessions   = ('is_weekend', 'sum')
    ).reset_index()
    total['last_session_recency_days'] = OBS_END_DAY - total['last_day']
    total['weekend_session_ratio']     = total['weekend_sessions'] / total['session_count']

    # Assemble
    feat = spine[['member_id']].copy()
    for s in [total[['member_id','session_count','active_days_app',
                      'last_session_recency_days','weekend_session_ratio']],
              sess_d1_3.reset_index(), sess_d4_6.reset_index(),
              sess_d1_7.reset_index(), sess_d8_14.reset_index(),
              var_stats]:
        feat = feat.merge(s, on='member_id', how='left')

    feat['session_trend_3d'] = feat['sess_d4_6'].fillna(0)  - feat['sess_d1_3'].fillna(0)
    feat['session_trend_7d'] = feat['sess_d8_14'].fillna(0) - feat['sess_d1_7'].fillna(0)
    feat['session_cv']       = (feat['session_std_daily']
                                / feat['session_count'].replace(0, np.nan)).fillna(0)

    out_cols = ['session_count', 'active_days_app', 'last_session_recency_days',
                'weekend_session_ratio', 'session_trend_3d', 'session_trend_7d',
                'session_std_daily', 'max_sessions_day', 'session_cv']
    feat[out_cols] = feat[out_cols].fillna(0)
    return feat[['member_id'] + out_cols]

app_feat_tr = build_app_features(app_tr, labels)
app_feat_te = build_app_features(app_te, test)
print('App features shape:', app_feat_tr.shape)
print(app_feat_tr.describe().round(3))

## 3. WellCo web visit features (domain-filtered)

Only `health.wellco` visits — 90.3% of raw visits are non-WellCo noise (portal.site, example.com, etc.).

In [ ]:
def build_wellco_features(web, spine):
    wellco = web[web['url'].str.contains('health.wellco', na=False)].copy()
    wellco['day']         = (wellco['timestamp'].dt.normalize() - OBS_START).dt.days + 1
    wellco['path_prefix'] = wellco['url'].str.extract(r'wellco/([^/]+)')

    d1_7  = (wellco[wellco['day'].between(1, 7)]
             .groupby('member_id').size().rename('wellco_d1_7'))
    d8_14 = (wellco[wellco['day'].between(8, 14)]
             .groupby('member_id').size().rename('wellco_d8_14'))

    agg = wellco.groupby('member_id').agg(
        wellco_visit_count          = ('day', 'count'),
        wellco_active_days          = ('day', 'nunique'),
        wellco_last_day             = ('day', 'max'),
        n_wellco_categories         = ('path_prefix', 'nunique')
    ).reset_index()
    agg['wellco_last_visit_recency_days'] = OBS_END_DAY - agg['wellco_last_day']

    feat = spine[['member_id']].copy()
    feat = feat.merge(agg[['member_id','wellco_visit_count','wellco_active_days',
                            'wellco_last_visit_recency_days','n_wellco_categories']],
                      on='member_id', how='left')
    feat = feat.merge(d1_7.reset_index(),  on='member_id', how='left')
    feat = feat.merge(d8_14.reset_index(), on='member_id', how='left')

    feat['wellco_visit_trend'] = feat['wellco_d8_14'].fillna(0) - feat['wellco_d1_7'].fillna(0)

    out_cols = ['wellco_visit_count','wellco_active_days',
                'wellco_last_visit_recency_days','n_wellco_categories','wellco_visit_trend']
    feat[out_cols] = feat[out_cols].fillna(0)
    # Members with no WellCo visits: recency = 14 (never visited)
    feat['wellco_last_visit_recency_days'] = np.where(
        feat['wellco_visit_count'] > 0, feat['wellco_last_visit_recency_days'], 14)
    return feat[['member_id'] + out_cols]

wellco_feat_tr = build_wellco_features(web_tr, labels)
wellco_feat_te = build_wellco_features(web_te, test)
print('WellCo features shape:', wellco_feat_tr.shape)

## 4. NLP features from web visits (all domains)

Two complementary NLP approaches applied to concatenated `title + description` text across **all** visits per member (not just WellCo domain — non-WellCo sites like `living.better` and `care.portal` also contain health content):

1. **Keyword category counts** — count of visits matching health topic lexicons drawn from the WellCo client brief (diabetes, hypertension, nutrition, exercise, sleep, stress, heart)
2. **LSA components** — TF-IDF (500 terms) + Truncated SVD (5 components) capturing latent health topic patterns; fitted on train, applied to test to prevent leakage

In [ ]:
# Health keyword taxonomy sourced from wellco_client_brief.txt
HEALTH_KEYWORDS = {
    'kw_diabetes':     ['diabetes', 'blood glucose', 'insulin', 'glycemic', 'hba1c'],
    'kw_hypertension': ['hypertension', 'blood pressure'],
    'kw_nutrition':    ['nutrition', 'diet', 'mediterranean', 'fiber', 'cholesterol',
                        'eating', 'meal', 'lipid'],
    'kw_exercise':     ['exercise', 'aerobic', 'cardio', 'strength', 'fitness',
                        'workout', 'physical activity'],
    'kw_sleep':        ['sleep', 'insomnia', 'restorative', 'sleep hygiene', 'sleep apnea'],
    'kw_stress':       ['stress', 'mindfulness', 'meditation', 'mental health',
                        'wellbeing', 'resilience'],
    'kw_heart':        ['heart', 'cardiac', 'cardiometabolic', 'cardiovascular'],
}
KW_COLS = list(HEALTH_KEYWORDS.keys())


def build_nlp_features(web, spine, vectorizer=None, svd=None, fit=False):
    """
    fit=True  → fit TF-IDF + SVD on this split (train call)
    fit=False → transform only using pre-fitted objects (test call)
    Returns (feature_df, vectorizer, svd)
    """
    web = web.copy()
    web['text'] = (
        web['title'].fillna('') + ' ' + web['description'].fillna('')
    ).str.lower()

    # --- Per-visit keyword match counts aggregated to member level ---
    kw_frames = []
    for col, keywords in HEALTH_KEYWORDS.items():
        pattern = '|'.join(keywords)
        match_counts = (web[web['text'].str.contains(pattern, na=False)]
                        .groupby('member_id').size().rename(col))
        kw_frames.append(match_counts)

    kw_df = pd.concat(kw_frames, axis=1).fillna(0).reset_index()
    kw_df['n_health_categories'] = (kw_df[KW_COLS] > 0).sum(axis=1)

    # Health content ratio: visits with any health keyword / total visits
    all_pattern = '|'.join(kw for kws in HEALTH_KEYWORDS.values() for kw in kws)
    web['_health'] = web['text'].str.contains(all_pattern, na=False)
    total_v  = web.groupby('member_id').size().rename('_total')
    health_v = web.groupby('member_id')['_health'].sum().rename('_health_v')
    ratio_df = pd.concat([total_v, health_v], axis=1).reset_index()
    ratio_df['health_content_ratio'] = ratio_df['_health_v'] / ratio_df['_total']
    kw_df = kw_df.merge(ratio_df[['member_id', 'health_content_ratio']],
                        on='member_id', how='left')

    # --- LSA: TF-IDF on per-member concatenated text ---
    member_text = (web.groupby('member_id')['text']
                   .apply(' '.join).reset_index()
                   .rename(columns={'text': 'all_text'}))

    if fit:
        vectorizer = TfidfVectorizer(max_features=500, stop_words='english', min_df=2)
        svd = TruncatedSVD(n_components=5, random_state=42)
        tfidf_mat = vectorizer.fit_transform(member_text['all_text'])
        lsa_mat   = svd.fit_transform(tfidf_mat)
    else:
        tfidf_mat = vectorizer.transform(member_text['all_text'])
        lsa_mat   = svd.transform(tfidf_mat)

    lsa_df = pd.DataFrame(lsa_mat, columns=[f'lsa_{i+1}' for i in range(5)])
    lsa_df['member_id'] = member_text['member_id'].values
    member_text = member_text.merge(lsa_df, on='member_id', how='left')

    # Join keyword + LSA features, then merge onto spine
    nlp_combined = member_text[['member_id'] + [f'lsa_{i+1}' for i in range(5)]]
    nlp_combined = nlp_combined.merge(kw_df, on='member_id', how='outer')

    nlp_cols = (KW_COLS + ['n_health_categories', 'health_content_ratio']
                + [f'lsa_{i+1}' for i in range(5)])

    feat = spine[['member_id']].copy()
    feat = feat.merge(nlp_combined[['member_id'] + nlp_cols], on='member_id', how='left')
    feat[nlp_cols] = feat[nlp_cols].fillna(0)

    return feat[['member_id'] + nlp_cols], vectorizer, svd


nlp_feat_tr, vectorizer, svd_model = build_nlp_features(web_tr, labels, fit=True)
nlp_feat_te, _,          _         = build_nlp_features(
    web_te, test, vectorizer=vectorizer, svd=svd_model, fit=False)

print('NLP features shape:', nlp_feat_tr.shape)
print('\nKeyword feature means (train):')
print(nlp_feat_tr[KW_COLS].mean().round(2))
print('\nLSA explained variance ratio:', svd_model.explained_variance_ratio_.round(3))

## 5. Claims features

Deduplicate on `(member_id, icd_code, diagnosis_date)` first (1,676 duplicates found in EDA). Binary flags for the three WellCo clinical ICD codes; noise flag for the 7 unexpected codes.

In [ ]:
EXPECTED_ICD = {'E11.9', 'I10', 'Z71.3'}

def build_claims_features(claims, spine):
    claims = claims.drop_duplicates(subset=['member_id', 'icd_code', 'diagnosis_date'])

    # Binary flags for each expected ICD code
    exp = claims[claims['icd_code'].isin(EXPECTED_ICD)]
    icd_flags = (exp.groupby(['member_id', 'icd_code'])
                    .size()
                    .unstack(fill_value=0)
                    .clip(upper=1))
    icd_flags.columns = [f'has_{c}' for c in icd_flags.columns]
    # Ensure all three columns exist even if a code has zero members
    for code in EXPECTED_ICD:
        col = f'has_{code}'
        if col not in icd_flags.columns:
            icd_flags[col] = 0

    total_claims = claims.groupby('member_id').size().rename('total_claims_deduped')
    noise_flag   = (claims[~claims['icd_code'].isin(EXPECTED_ICD)]
                    .groupby('member_id').size().gt(0).rename('has_noise_icd'))

    icd_cols = ['has_E11.9', 'has_I10', 'has_Z71.3']
    feat = spine[['member_id']].copy()
    feat = feat.merge(icd_flags[icd_cols].reset_index(), on='member_id', how='left')
    feat = feat.merge(total_claims.reset_index(),         on='member_id', how='left')
    feat = feat.merge(noise_flag.reset_index(),           on='member_id', how='left')

    for col in icd_cols + ['total_claims_deduped', 'has_noise_icd']:
        feat[col] = feat[col].fillna(0)
    feat['n_clinical_conditions'] = feat[icd_cols].sum(axis=1)

    out_cols = icd_cols + ['n_clinical_conditions', 'total_claims_deduped', 'has_noise_icd']
    return feat[['member_id'] + out_cols]

claims_feat_tr = build_claims_features(claims_tr, labels)
claims_feat_te = build_claims_features(claims_te, test)
print('Claims features shape:', claims_feat_tr.shape)
print(claims_feat_tr.describe().round(3))

## 6. Assemble & validate

In [ ]:
def assemble(spine, labels_df, feature_frames):
    df = spine[['member_id']].copy()
    for ff in feature_frames:
        df = df.merge(ff, on='member_id', how='left')
    if labels_df is not None:
        df = df.merge(labels_df[['member_id', 'churn', 'outreach']],
                      on='member_id', how='left')
    return df

feature_frames_tr = [tenure_tr, app_feat_tr, wellco_feat_tr, nlp_feat_tr, claims_feat_tr]
feature_frames_te = [tenure_te, app_feat_te, wellco_feat_te, nlp_feat_te, claims_feat_te]

train_df = assemble(labels, labels,  feature_frames_tr)
test_df  = assemble(test,   None,    feature_frames_te)

# Validation
print('=== Train ===')
print('Shape:', train_df.shape)
print('Nulls:', train_df.isnull().sum().sum())
print('Churn rate:', train_df['churn'].mean().round(4))

print('\n=== Test ===')
print('Shape:', test_df.shape)
print('Nulls:', test_df.isnull().sum().sum())

feature_cols = [c for c in train_df.columns if c not in ['member_id', 'churn', 'outreach']]
print(f'\nFeature count: {len(feature_cols)}')
print('Features:', feature_cols)

## 7. Feature distributions & correlation with churn

In [ ]:
import matplotlib.pyplot as plt

corr = train_df[feature_cols + ['churn']].corr()['churn'].drop('churn').sort_values()

fig, ax = plt.subplots(figsize=(6, 10))
corr.plot(kind='barh', ax=ax, color=['tomato' if v > 0 else 'steelblue' for v in corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature correlation with churn')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=120)
plt.show()
print(corr.round(4))

## 8. Save

In [ ]:
# Cast boolean/object columns to int for clean parquet storage
for df in [train_df, test_df]:
    for col in df.select_dtypes(include=['bool','object']).columns:
        if col != 'member_id':
            df[col] = df[col].astype(int)

train_df.to_parquet('train_features.parquet', index=False)
test_df.to_parquet('test_features.parquet',   index=False)
print('Saved train_features.parquet:', train_df.shape)
print('Saved test_features.parquet: ', test_df.shape)